# RisiKo RL — Training Bot

Notebook di training per il bot RL di RisiKo, da usare su **Colab Pro**.

**Workflow:**
1. Cella 1: monta Google Drive e clona il repo da GitHub
2. Cella 2: installa dipendenze RL
3. Cella 3: verifica che il simulatore funzioni
4. Cella 4: configura il training
5. Cella 5: lancia il training (può durare ore)
6. Cella 6: valuta il bot trainato
7. Cella 7 (opzionale): carica un modello esistente

**IMPORTANTE:** prima di iniziare, sostituisci `IL_TUO_USERNAME` e `il-tuo-repo` nella cella 1 con i tuoi dati GitHub.

## Cella 1: Setup Drive e clona repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/risiko-rl-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

# Clona il repo
%cd /content
!rm -rf risiko-rl
!git clone https://github.com/IL_TUO_USERNAME/il-tuo-repo.git risiko-rl
%cd risiko-rl

## Cella 2: Installa dipendenze

In [ ]:
!pip install -r requirements.txt -q
!pip install sb3-contrib -q
!pip install torch -q

import torch
import gymnasium as gym
from sb3_contrib import MaskablePPO
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'Gymnasium: {gym.__version__}')

## Cella 3: Verifica simulatore

In [ ]:
for f in ['test_data', 'test_setup', 'test_motore', 'test_sdadata',
          'test_partita_completa', 'test_encoding', 'test_azioni', 'test_env']:
    print(f'=== {f} ===')
    !python tests/{f}.py 2>&1 | tail -3

## Cella 4: Configura training

In [ ]:
import sys
sys.path.insert(0, '/content/risiko-rl')

from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback
import numpy as np
import torch

from risiko_env import RisikoEnv
from risiko_env import encoding as _encoding

# === OPZIONE B: validate metodologia BEST MODEL prima di Stage A ===
# Disabilita Stage A per questo training: observation 318 feature (come baseline 500k).
# Cambieremo a True solo nel prossimo training (Stage A vero).
_encoding.STAGE_A_ATTIVO = False
print(f'Stage A: {_encoding.STAGE_A_ATTIVO} (observation: {_encoding.get_dim_observation()} feature)')

def mask_fn(env):
    return env.get_action_mask()

def make_env(seed):
    def _init():
        # IMPORTANTE: ogni subprocess re-importa il modulo, quindi
        # impostiamo il flag QUI (non solo nel main process).
        from risiko_env import encoding as _encoding
        _encoding.STAGE_A_ATTIVO = False  # v4.1b: no Stage A
        env = RisikoEnv(seed=seed)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

# === IPERPARAMETRI v5-STABLE (priorita: stabilita, NO Stage A) ===
# Diagnosi v3: bot accumulava in un territorio, faceva turni passivi, attaccava kamikaze.
# v5-stable: priorita' alla STABILITA' della pipeline, non al massimo win rate.
#
# Storia recente:
# - Baseline 500k (v4.1-test 1M): 29% WR validato
# - v4.1-test 3M: degradato a 19% WR (over-training)
# - v4.1b 1M con EvalCallback: best=18% (overfit), checkpoint 700k=26% — instabile
#
# Diagnosi: pipeline produce risultati con varianza ±5-7%. Per Stage A
# serve replicabilita' entro ±2%.
#
# Fix v5-stable:
# - lr 2e-4 -> 1e-4: piu' lento ma stabile
# - n_steps 1024 -> 2048: batch piu' grandi, gradient piu' puliti
# - batch_size 64 -> 128
# - ent_coef 0.02 -> 0.05: piu' esplorazione per evitare minimi locali
# - 2M step (vs 1M): piu' tempo per stabilizzare
# - EvalCallback: 4 env con seed diversi, 20 eps eval, ogni 200k step
#
# Criterio successo:
# - best_model validato 500 partite >= 28%
# - nessun crollo a < 15% durante training
# - varianza tra checkpoint vicini ridotta

TOTAL_TIMESTEPS = 2_000_000  # v5-stable: 2M con iperparametri piu' stabili
N_ENVS = 8
SEED_BASE = 42

env = SubprocVecEnv([make_env(SEED_BASE + i) for i in range(N_ENVS)])
env = VecMonitor(env)

model = MaskablePPO(
    'MlpPolicy',
    env,
    learning_rate=1e-4,      # v5-stable: piu' lento, meno instabilita'
    n_steps=2048,            # v5-stable: batch piu' grandi, gradient piu' puliti
    batch_size=128,          # v5-stable: batch_size piu' grande
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.05,           # v5-stable: piu' esplorazione per evitare minimi locali
    verbose=1,
    tensorboard_log=f'{CHECKPOINT_DIR}/tensorboard/',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print(f'Modello creato. Device: {model.device}')
print(f'Parametri rete: {sum(p.numel() for p in model.policy.parameters()):,}')
print(f'Iperparametri: lr={model.learning_rate}, n_steps={model.n_steps}, ent_coef={model.ent_coef}')


## Cella 5: Lancia training

In [ ]:
# === CHECKPOINT + EARLY-STOPPING + BEST MODEL TRACKING ===
# Lezione del v4.1: il bot a 1M era migliore di quello a 3M.
# Quindi: salviamo checkpoint frequenti E un "best_model" basato su valutazione periodica.

from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from sb3_contrib.common.wrappers import ActionMasker

# Salva checkpoint regolari (ogni 50k step)
checkpoint_callback = CheckpointCallback(
    save_freq=50_000 // N_ENVS,
    save_path=CHECKPOINT_DIR,
    name_prefix='risiko_bot',
)

# Env di valutazione (1 sola partita per volta, con action masking)
def make_eval_env(seed):
    # Vedi nota sopra: imposta flag in subprocess.
    def _init():
        from risiko_env import encoding as _encoding
        _encoding.STAGE_A_ATTIVO = False  # v5-stable: no Stage A
        env = RisikoEnv(seed=seed)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

# v5-stable: 4 env eval con seed diversi (10000, 20000, 30000, 40000)
# Cosi' la valutazione vede partite diverse, non sempre la stessa.
EVAL_SEEDS = [10000, 20000, 30000, 40000]
eval_env = SubprocVecEnv([make_eval_env(s) for s in EVAL_SEEDS])
eval_env = VecMonitor(eval_env)

# Salva automaticamente il MIGLIOR modello trovato durante il training
# (basato su reward medio in 20 partite di valutazione, ogni 100k step)
# v5-stable: EvalCallback robusta (no overfit al seed)
# - 20 episodi per misurazione affidabile (std > 0)
# - eval ogni 200k step (meno frequente ma piu' affidabile)
# - deterministic=False per esplorare seed diversi
# - VERBOSE: stampa anche +/- std per capire se la valutazione e' stabile
best_model_callback = MaskableEvalCallback(
    eval_env,
    best_model_save_path=CHECKPOINT_DIR + '/best/',
    log_path=CHECKPOINT_DIR + '/eval_logs/',
    eval_freq=200_000 // N_ENVS,  # ogni 200k step (era 100k)
    n_eval_episodes=20,
    deterministic=True,           # deterministic ok, ma su 20 partite con seed diversi
    verbose=1,
)

print('Callback configurate:')
print(f'  - Checkpoint ogni 50k step in {CHECKPOINT_DIR}')
print(f'  - Best model in {CHECKPOINT_DIR}/best/ (auto-aggiornato ogni 100k step)')
print(f'  - Valutazione su 20 partite, deterministica')

import time
inizio = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[checkpoint_callback, best_model_callback],
    progress_bar=True,
)
durata = time.time() - inizio
print(f'\nTraining completato in {durata/60:.1f} minuti')

# Salva anche il modello finale (utile per debug)
model.save(f'{CHECKPOINT_DIR}/risiko_bot_finale')
print(f'Modello finale salvato.')
print(f'\nIMPORTANTE: per la valutazione USA il file in {CHECKPOINT_DIR}/best/best_model.zip')
print(f'Il modello finale (risiko_bot_finale.zip) potrebbe essere PEGGIORE del best.')

## Cella 6: Valuta il bot trainato

In [ ]:
from risiko_env import RisikoEnv
import numpy as np

def valuta_bot(model, n_partite=100, verbose=False):
    vittorie = 0
    rewards_totali = []
    durate_step = []

    for seed in range(n_partite):
        env = RisikoEnv(seed=seed)
        obs, info = env.reset()
        terminated = False
        truncated = False
        step_count = 0

        while not (terminated or truncated):
            mask = info['action_mask']
            action, _ = model.predict(obs, action_masks=mask, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            step_count += 1

        rewards_totali.append(reward)
        durate_step.append(step_count)
        if reward == 1.0:
            vittorie += 1

        if verbose and seed < 5:
            print(f'Partita {seed}: reward={reward}, step={step_count}, '
                  f'vincitore={info["vincitore"]}')

    win_rate = vittorie / n_partite
    reward_medio = np.mean(rewards_totali)
    step_medio = np.mean(durate_step)

    return {
        'win_rate': win_rate,
        'reward_medio': reward_medio,
        'step_medio': step_medio,
    }

risultati = valuta_bot(model, n_partite=100, verbose=True)
print()
print(f'Win rate: {risultati["win_rate"]*100:.1f}% (atteso ~25% per bot random)')
print(f'Reward medio: {risultati["reward_medio"]:.3f}')
print(f'Step medio: {risultati["step_medio"]:.0f}')

## Cella 7 (opzionale): Carica modello esistente

In [ ]:
import glob
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/risiko_bot_*.zip'))
if checkpoints:
    print(f'Checkpoints disponibili:')
    for c in checkpoints:
        print(f'  {c}')
    model_loaded = MaskablePPO.load(checkpoints[-1], env=env)
    print(f'\nCaricato: {checkpoints[-1]}')
else:
    print('Nessun checkpoint trovato')